<a href="https://colab.research.google.com/github/HarithaGottumukkala/data266-1598/blob/main/cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA 266 - Homework 1
## Question 3 - CUDA Matrix Multiplication

### Personal Parameters

- SID4 = 1598
- SEED = 1598
- SLICE = 598
- HP_ID = 2
- CLS_A = 8
- CLS_B = 5

In this part of the homework, I will implement matrix multiplication using CUDA C. I will compare the execution time of a CPU implementation with a GPU implementation and study how CUDA blocks and threads are used to perform matrix multiplication in parallel.

In [1]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp_ky45qux".


## 3.1 Checking the CUDA Setup

Before implementing matrix multiplication, I first want to make sure that a basic CUDA program can be compiled and executed correctly on the GPU.

In CUDA, the normal `main()` function starts on the CPU, which is called the host. A function marked with `__global__` is called a kernel, and that function runs on the GPU, which is called the device.

I will start with one very small GPU program. This is only a setup check before moving to the actual matrix multiplication implementation.

In [ ]:
%%writefile hello_cuda.cu

#include <stdio.h>

__global__ void helloFromGPU()
{
    printf("Hello from GPU!\n");
}

int main()
{
    printf("Hello from CPU!\n");

    helloFromGPU<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}

Writing hello_cuda.cu


In [ ]:
!nvcc -arch=sm_75 hello_cuda.cu -o hello_cuda

In [ ]:
!./hello_cuda

Hello from CPU!
Hello from GPU!


## 3.2 Understanding GPU Threads

The previous example used only one GPU thread. However, the main advantage of CUDA is that many threads can run at the same time.

Each thread inside a CUDA block has its own thread index. CUDA provides this index through `threadIdx`.

In this small example, I will launch 10 GPU threads and check the index of each thread. I will print a message only from thread number 5. This helps me understand how an individual thread can be identified before using threads for matrix multiplication.

In [ ]:
%%writefile threadindex.cu

#include <stdio.h>

__global__ void helloFromGPU(void)
{
    // Get the index of the current thread
    int thread_id = threadIdx.x;

    // Only thread number 5 prints the message
    if (thread_id == 5)
    {
        printf("Hello World from GPU thread %d!\n", thread_id);
    }
}

int main(void)
{
    printf("Hello World from CPU!\n");

    // 1 block with 10 threads
    helloFromGPU<<<1, 10>>>();

    cudaDeviceReset();

    return 0;
}

Writing threadindex.cu


In [ ]:
!nvcc -arch=sm_75 threadindex.cu -o threadindex

In [ ]:
!./threadindex

Hello World from CPU!
Hello World from GPU thread 5!


## 3.3 Moving Data Between CPU and GPU

So far, I have learned how a CUDA kernel runs on the GPU and how individual GPU threads can be identified.

The next thing I need to understand is memory. The CPU and GPU have separate memory spaces. A variable created normally in the program belongs to the CPU, but a CUDA kernel needs data that is stored in GPU memory.

In this example, I will add two integers using the GPU. I will first create the values on the CPU, allocate memory for them on the GPU, copy the input values from the CPU to the GPU, run the CUDA kernel, and finally copy the result back to the CPU.

This same sequence will later be used for matrix multiplication, except that instead of copying two integers, I will copy complete matrices.

In [ ]:
%%writefile add_integers.cu

#include <stdio.h>

__global__ void add(int *a, int *b, int *c)
{
    *c = *a + *b;
}

int main(void)
{
    int a, b, c;             // CPU copies
    int *d_a, *d_b, *d_c;    // GPU copies

    int size = sizeof(int);

    // Allocate memory on the GPU
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Values stored on the CPU
    a = 2;
    b = 7;

    // Copy input values from CPU to GPU
    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    // Run the addition kernel on the GPU
    add<<<1, 1>>>(d_a, d_b, d_c);

    // Copy the result from GPU back to CPU
    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);

    printf("The result of %d + %d = %d\n", a, b, c);

    // Release GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing add_integers.cu


In [ ]:
!nvcc -arch=sm_75 add_integers.cu -o add_integers

In [ ]:
!./add_integers

The result of 2 + 7 = 9


## 3.4 CUDA Matrix Multiplication

After testing a basic CUDA kernel and learning how data is copied between the CPU and GPU, I can now move to the main CUDA task in this assignment.

For matrix multiplication, I have two input matrices A and B and I want to calculate the output matrix C.

Instead of calculating every element of C one after another on the CPU, I will use CUDA blocks and threads so that many elements of the output matrix can be calculated in parallel on the GPU.

I will use a two-dimensional block of threads. Each GPU thread will be responsible for calculating one element of the output matrix. The row and column handled by a thread are found using its block index and thread index.

I will first implement both a CPU version and a CUDA GPU version. After checking that they produce the same result, I will measure their execution times for matrix sizes 256, 1024, and 4096.

### 3.4.1 Blocks and Threads Used for the Matrix

I am using blocks of 16 × 16 threads. This means that each block contains 256 threads.

Each thread calculates one element of matrix C.

The column handled by a thread is calculated using:

`blockIdx.x * blockDim.x + threadIdx.x`

The row is calculated using:

`blockIdx.y * blockDim.y + threadIdx.y`

For example, if a thread is responsible for row 2 and column 3, that thread calculates C[2][3]. To calculate this value, it multiplies the values from row 2 of matrix A with the corresponding values from column 3 of matrix B and adds them together.

Since the matrices can be larger than one block, I will create a two-dimensional grid containing enough blocks to cover the entire matrix.

In [2]:
%%writefile matrix_multiplication.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <chrono>
#include <cmath>
#include <algorithm>

// CUDA kernel for matrix multiplication
__global__ void matrixMultiplyGPU(
    const float *A,
    const float *B,
    float *C,
    int N)
{
    // Find the row and column handled by this thread
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Make sure the thread is inside the matrix
    if (row < N && col < N)
    {
        float sum = 0.0f;

        // Multiply one row of A with one column of B
        for (int k = 0; k < N; k++)
        {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}


// CPU version of matrix multiplication
void matrixMultiplyCPU(
    const float *A,
    const float *B,
    float *C,
    int N)
{
    // Start the output matrix with zeros
    std::fill(C, C + (long long)N * N, 0.0f);

    for (int i = 0; i < N; i++)
    {
        for (int k = 0; k < N; k++)
        {
            float a = A[i * N + k];

            for (int j = 0; j < N; j++)
            {
                C[i * N + j] += a * B[k * N + j];
            }
        }
    }
}


// Fill matrices with deterministic values
void initializeMatrix(float *matrix, int N, int offset)
{
    long long total = (long long)N * N;

    for (long long i = 0; i < total; i++)
    {
        matrix[i] = ((i + offset) % 100) / 100.0f;
    }
}


// Run the experiment for one matrix size
void runExperiment(int N)
{
    printf("\n========================================\n");
    printf("Matrix size: %d x %d\n", N, N);
    printf("========================================\n");

    long long elements = (long long)N * N;
    size_t bytes = elements * sizeof(float);

    // -----------------------------
    // Allocate CPU memory
    // -----------------------------
    float *h_A = (float *)malloc(bytes);
    float *h_B = (float *)malloc(bytes);
    float *h_C_CPU = (float *)malloc(bytes);
    float *h_C_GPU = (float *)malloc(bytes);

    initializeMatrix(h_A, N, 1);
    initializeMatrix(h_B, N, 7);

    // -----------------------------
    // CPU timing
    // -----------------------------
    auto cpu_start = std::chrono::high_resolution_clock::now();

    matrixMultiplyCPU(h_A, h_B, h_C_CPU, N);

    auto cpu_end = std::chrono::high_resolution_clock::now();

    double cpu_time =
        std::chrono::duration<double, std::milli>(
            cpu_end - cpu_start
        ).count();


    // -----------------------------
    // Allocate GPU memory
    // -----------------------------
    float *d_A, *d_B, *d_C;

    cudaMalloc((void **)&d_A, bytes);
    cudaMalloc((void **)&d_B, bytes);
    cudaMalloc((void **)&d_C, bytes);


    // CUDA events used for timing
    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);


    // -----------------------------
    // Host-to-Device transfer timing
    // -----------------------------
    cudaEventRecord(start);

    cudaMemcpy(
        d_A,
        h_A,
        bytes,
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_B,
        h_B,
        bytes,
        cudaMemcpyHostToDevice
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float h2d_time = 0.0f;

    cudaEventElapsedTime(
        &h2d_time,
        start,
        stop
    );


    // -----------------------------
    // CUDA block and grid setup
    // -----------------------------
    dim3 block(16, 16);

    dim3 grid(
        (N + block.x - 1) / block.x,
        (N + block.y - 1) / block.y
    );


    // -----------------------------
    // Warm-up GPU kernel
    // -----------------------------
    matrixMultiplyGPU<<<grid, block>>>(
        d_A,
        d_B,
        d_C,
        N
    );

    cudaDeviceSynchronize();


    // -----------------------------
    // GPU kernel timing
    // -----------------------------
    cudaEventRecord(start);

    matrixMultiplyGPU<<<grid, block>>>(
        d_A,
        d_B,
        d_C,
        N
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float kernel_time = 0.0f;

    cudaEventElapsedTime(
        &kernel_time,
        start,
        stop
    );


    // -----------------------------
    // Device-to-Host transfer timing
    // -----------------------------
    cudaEventRecord(start);

    cudaMemcpy(
        h_C_GPU,
        d_C,
        bytes,
        cudaMemcpyDeviceToHost
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float d2h_time = 0.0f;

    cudaEventElapsedTime(
        &d2h_time,
        start,
        stop
    );


    // -----------------------------
    // Correctness check
    // -----------------------------
    float max_error = 0.0f;

    for (long long i = 0; i < elements; i++)
    {
        float error =
            fabs(h_C_CPU[i] - h_C_GPU[i]);

        if (error > max_error)
        {
            max_error = error;
        }
    }


    // -----------------------------
    // Final timing calculations
    // -----------------------------
    float transfer_time =
        h2d_time + d2h_time;

    float gpu_end_to_end =
        transfer_time + kernel_time;

    double speedup =
        cpu_time / gpu_end_to_end;


    // -----------------------------
    // Print results
    // -----------------------------
    printf("CPU time: %.3f ms\n", cpu_time);

    printf(
        "GPU kernel time: %.3f ms\n",
        kernel_time
    );

    printf(
        "H2D time: %.3f ms\n",
        h2d_time
    );

    printf(
        "D2H time: %.3f ms\n",
        d2h_time
    );

    printf(
        "H2D + D2H time: %.3f ms\n",
        transfer_time
    );

    printf(
        "GPU end-to-end time: %.3f ms\n",
        gpu_end_to_end
    );

    printf(
        "End-to-end speedup: %.3fx\n",
        speedup
    );

    printf(
        "Maximum error: %.6f\n",
        max_error
    );


    // -----------------------------
    // Cleanup
    // -----------------------------
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C_CPU);
    free(h_C_GPU);
}


int main(int argc, char **argv)
{
    // If a size is supplied, run only that size.
    // This will be useful later for profiling.
    if (argc == 2)
    {
        int N = atoi(argv[1]);
        runExperiment(N);
        return 0;
    }

    // Required matrix sizes for HW1
    int sizes[] = {
        256,
        1024,
        4096
    };

    for (int i = 0; i < 3; i++)
    {
        runExperiment(sizes[i]);
    }

    return 0;
}

Writing matrix_multiplication.cu


In [3]:
!nvcc -O3 -arch=sm_75 matrix_multiplication.cu -o matrix_multiplication

In [4]:
!./matrix_multiplication 256


Matrix size: 256 x 256
CPU time: 2.732 ms
GPU kernel time: 0.099 ms
H2D time: 0.716 ms
D2H time: 0.200 ms
H2D + D2H time: 0.915 ms
GPU end-to-end time: 1.014 ms
End-to-end speedup: 2.694x
Maximum error: 0.000015
